# Rome in a Day: A City Hidden in Photos

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_title_frontispiece.jpg" alt="Piranesi&#x27;s frontispiece to the Views of Rome: Minerva among Roman ruins (etching, 1748)" title="Piranesi&#x27;s frontispiece to the Views of Rome: Minerva among Roman ruins (etching, 1748)" width="300" align="right" style="margin: 4px 0 10px 18px">

*ES666: Structure from Motion at city scale*

Elena works on 3D maps of historic buildings. Marco has recently joined her lab, and you are helping them prepare a
demo for the next morning: how ordinary tourist photos can become a 3D model. The folder holds thousands of
unlabelled images of fountains, churches, squares and carvings, plus about 2,000 that are not from Rome at all.

**Marco:** "A model of the whole city?"

**Elena:** "Let's start by finding which photos belong together. Then we'll build the Trevi Fountain and try joining
two maps of St Peter's. Rome has been mapped by hand before:
[Giambattista Nolli](https://en.wikipedia.org/wiki/Giambattista_Nolli) surveyed it from 1736 for a plan published in
1748, and Piranesi engraved its views. We'll do their job from other people's photos."

This is **Structure from Motion** (SfM): using photos taken from different viewpoints to estimate the shape of a
scene and the position and orientation of its cameras. The hard part is the number of photos: thousands need a way
to find matches without comparing every pair. This is a small version of
[Building Rome in a Day](https://grail.cs.washington.edu/rome/) (Agarwal, Snavely, Simon, Seitz and Szeliski, 2009),
which did this for a large collection of internet photos.

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/route.png" alt="TAV. I, From photos to a 3D Rome: a plan of Rome in Nolli&#x27;s manner, with a route through thirteen stations from the folder to your own photos; red stations are your tasks" title="TAV. I, From photos to a 3D Rome: a plan of Rome in Nolli&#x27;s manner, with a route through thirteen stations from the folder to your own photos; red stations are your tasks" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. I, From photos to a 3D Rome: a plan of Rome in Nolli&#x27;s manner, with a route through thirteen stations from the folder to your own photos; red stations are your tasks</i></sub></p>

You will write nine functions:

| Task | Function | Chapter | What it helps us do |
|---|---|---|---|
| 1 | `tfidf_matrix` | 1 | Describe each photo compactly by its visual features. |
| 2 | `top_k_neighbours` | 1 | Find the most promising photos to compare with it. |
| 3 | `match_descriptors` | 2 | Match features while rejecting ambiguous choices. |
| 4 | `query_expansion` | 2 | Find more candidates through photos already linked. |
| 5 | `connected_components` | 3 | Split the collection into groups of connected photos. |
| 6 | `homography_dlt` | 4 | Fit a 3 by 3 image transformation to matching points. |
| 7 | `homography_ratio` | 4 | Check whether a pair is promising for recovering depth. |
| 8 | `triangulate_points` | 4 | Turn a point seen in two photos into a 3D point. |
| 9 | `umeyama` | 5 | Align two 3D maps in scale, rotation and position. |

Two of them also work inside the toolbox in Chapter 4: your `match_descriptors` builds the Trevi feature tracks, and
your `homography_ratio` helps choose the starting pair. In Chapter 7 (Task 10) seven of them rebuild a place you
photograph yourself.

## Getting started

The setup cell gets the toolbox from the course repository (with `git clone`, on Colab) and downloads the photo
collection (about 580 MB) the first time; later runs reuse them.
If your instructor shares the link to a smaller collection (about 170 MB, the same pipeline with fewer photos), you
can paste it into the setup cell instead; this is optional.

In [ ]:
# Local: clone the repository, `uv sync`, then `uv run jupyter lab` in it (the toolbox/ folder is already there).
# Colab: everything is preinstalled; if an import fails, run  !pip install gdown scikit-image
import os
import re
import shutil
import time
import zipfile
import zlib


def fetch_zip(url, name):
    """The zip file name, downloaded from url unless a complete copy is already here. The download goes into
    name + ".part" and is renamed to name only when it is complete; if it stops, the next run continues from the
    .part file (an HTTP Range request) instead of starting over."""
    if os.path.isfile(name):
        try:
            zipfile.ZipFile(name).close()  # opens only if the download finished
            return name
        except zipfile.BadZipFile:
            os.remove(name)
    if not re.match(r"https?://", url):
        raise RuntimeError(f"DATA_URL is not a link ({url}): paste the link you were given, or put the # back in "
                           "front of that line")
    import urllib.error
    import urllib.request
    part = name + ".part"
    done = os.path.getsize(part) if os.path.isfile(part) else 0
    request = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            if done and response.status != 206:  # the server sent the whole file again: start over
                done = 0
            total = done + int(response.headers.get("Content-Length") or 0)
            got, shown = done, 0.0
            with open(part, "ab" if done else "wb") as out:
                while True:
                    chunk = response.read(1 << 20)
                    if not chunk:
                        break
                    out.write(chunk)
                    got += len(chunk)
                    if time.time() - shown > 2:
                        print(f"\r{name}: {got / 1e6:,.0f} of {total / 1e6:,.0f} MB", end="", flush=True)
                        shown = time.time()
        print(f"\r{name}: {got / 1e6:,.0f} MB downloaded" + " " * 20)
        if total and got < total:
            raise OSError(f"the connection closed after {got:,} of {total:,} bytes")
    except urllib.error.HTTPError as err:
        if not (err.code == 416 and done):  # 416: the .part file already holds the whole file
            raise RuntimeError(f"the download of {name} failed ({err}): check the link and run this cell again") \
                from err
    except (urllib.error.URLError, OSError) as err:
        raise RuntimeError(f"the download of {name} stopped ({err}): run this cell again, it continues where it "
                           "stopped") from err
    if not zipfile.is_zipfile(part):  # a web page (a login, say) or a broken file instead of the zip
        os.remove(part)
        raise RuntimeError(f"the download of {name} is not a zip file, so it was removed: check the link (DATA_URL) "
                           "and run this cell again")
    os.replace(part, name)
    return name


def unzip(name, top, dest):
    """Unpack the zip name into a temporary folder, then move its top level folder top to dest in one step, so dest
    exists only when it is complete. A damaged zip is removed, so the next run downloads it again."""
    tmp = dest + ".unzipping"
    shutil.rmtree(tmp, ignore_errors=True)
    try:
        with zipfile.ZipFile(name) as zf:
            zf.extractall(tmp)
        os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
        os.replace(os.path.join(tmp, top), dest)
    except (zipfile.BadZipFile, zlib.error, EOFError) as err:
        shutil.rmtree(tmp, ignore_errors=True)
        os.remove(name)
        raise RuntimeError(f"{name} is damaged ({err}), so it was removed: run this cell again") from err
    except OSError as err:  # a full disk, say: the zip itself is fine, keep it
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError(f"{name} could not be unpacked ({err}): if the disk is full, free some space; then run "
                           "this cell again (the zip is reused)") from err
    shutil.rmtree(tmp, ignore_errors=True)
    os.remove(name)


REPO = "https://github.com/pikabell/es666-rome-in-a-day"  # the course repository: the toolbox and rome_from_folder.py
TOOLBOX_TAG = "toolbox-63efbeb30581"  # the release of the toolbox this notebook was made with
need_toolbox = not os.path.isfile(os.path.join("toolbox", "__init__.py"))
need_script = not os.path.isfile("rome_from_folder.py")  # only Chapter 7 (your own photos) uses it
if need_toolbox and os.path.isdir("toolbox"):
    raise RuntimeError("the toolbox folder is incomplete (an earlier copy was cut off): delete the toolbox folder "
                       "and run this cell again")
if need_toolbox or need_script:
    import subprocess
    import tempfile
    clone = tempfile.mkdtemp()  # a new folder of its own: nothing of yours is touched
    part = None
    try:
        if shutil.which("git") is None:
            raise RuntimeError(f"git is not installed: install it (https://git-scm.com), or clone {REPO} and open "
                               "the notebook from that folder")
        try:
            subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "--depth", "1", "--quiet",
                            "--branch", TOOLBOX_TAG, REPO, clone], check=True)
        except subprocess.CalledProcessError as err:
            raise RuntimeError(f"could not get the toolbox from {REPO} ({TOOLBOX_TAG}): check the internet "
                               "connection and run this cell again; if it keeps failing, tell your "
                               "instructor") from err
        part = tempfile.mkdtemp(dir=".", prefix=".setup-")  # files are copied here first, then moved in whole
        if need_toolbox:
            shutil.copytree(os.path.join(clone, "toolbox"), os.path.join(part, "toolbox"))
            os.replace(os.path.join(part, "toolbox"), "toolbox")
        if need_script:
            shutil.copy(os.path.join(clone, "rome_from_folder.py"), part)
            os.replace(os.path.join(part, "rome_from_folder.py"), "rome_from_folder.py")
    except RuntimeError:
        if need_toolbox:
            raise
        print(f"note: rome_from_folder.py is missing and could not be fetched from {REPO} (is git installed, and are "
              "you online?). Everything works without it except Chapter 7 (your own photos in 3D): run this cell "
              "again to fetch it.")
    finally:
        shutil.rmtree(clone, ignore_errors=True)
        if part is not None:
            shutil.rmtree(part, ignore_errors=True)

DATA_DIR = "data"
DATA_URL = "https://github.com/pikabell/es666-rome-in-a-day/releases/download/data-1/mini_rome.zip"  # about 580 MB
# Optional: if your instructor shares a direct link to the small collection (mini_rome_small.zip, about 170 MB),
# paste it here and remove the # in front:
# DATA_URL = "PASTE_SMALL_COLLECTION_LINK_HERE"
CRATES = ["mini_rome", "mini_rome_small"]


def is_complete(folder):
    return all(os.path.isfile(os.path.join(folder, f)) for f in ("meta/images.csv", "features/words.npz"))


def find_crate():
    for base in [DATA_DIR, os.path.join(DATA_DIR, "data")]:
        for name in CRATES:
            folder = os.path.join(base, name)
            if is_complete(folder):
                return folder
            if os.path.isdir(folder):
                raise RuntimeError(f"{folder} is incomplete (an earlier unzip was cut off): delete the folder "
                                   f"{folder} and run this cell again (a finished zip download is reused)")
    return None


ROME_DIR = find_crate()
if ROME_DIR is None:
    zips = [z for z in (n + ".zip" for n in CRATES) if os.path.isfile(z)] or ["mini_rome.zip"]
    name = fetch_zip(DATA_URL, zips[0])
    top = zipfile.ZipFile(name).namelist()[0].split("/")[0]  # mini_rome or mini_rome_small
    unzip(name, top, os.path.join(DATA_DIR, top))
    ROME_DIR = find_crate()
print("photo collection:", ROME_DIR)

## The toolbox

The toolbox provides feature extraction, geometric verification, camera pose estimation, bundle
adjustment, the maximum spanning tree and plotting (and its own triangulation, for the pipeline). Each photo has its
own calibration matrix `K`, since the images come from different cameras and zoom settings.

| Chapter | From the toolbox |
|---|---|
| Start | `load_mini_rome`, `show_crate` |
| 1 | `show_words`, `plot_similarity_matrix`, `show_neighbours` |
| 1 | `retrieval_precision`, `plot_retrieval_precision` |
| 2 | `neighbour_pairs`, `photos_of`, `sift_features`, `show_match_filters` |
| 2 | `show_word_matches`, `verify_pairs`, `plot_match_graph` |
| 3 | `UnionFind`, `show_components`, `component_purity` |
| 4 | `component_of_landmark`, `component_graph`, `maximum_spanning_tree`, `plot_spanning_tree`, `landmark_scene` |
| 4 | `strongest_pairs`, `pair_matches`, `show_homography_test`, `triangulation_pair`, `pair_cameras` |
| 4 | `show_reprojection`, `baseline_experiment`, `run_sfm_skeleton`, `model_pair`, `compare_with_model` |
| 4 | `register_rest`, `plot_reconstruction`, `densify` |
| 5 | `camera_centres_vs_gt`, `plot_alignment`, `st_peters_maps`, `plot_merge` |
| 5 | `densify_survey`, `plot_dense_merge` |
| 6 | `show_atlas` |
| Geometry | `hartley_normalize`, `eight_point`, `ransac_fundamental`, `projection_matrix`, `choose_pose` |
| Geometry | `pnp_ransac`, `bundle_adjust`, ... |
| Checks | `test_<name>()` for each of your functions, `ref.<name>` (the library versions) |

The toolbox calls your functions by name, so you can import it before writing them. `help(verify_pairs)` (or any
other name) shows what a tool does.

**How to work.** For each task, write the function, run its check cell, then the cells that use it; rerun a
function's cell after changing it. A cell that needs an unwritten function stops with `NotImplementedError`.

**Stuck?** `tfidf_matrix = ref.tfidf_matrix` (and so on) borrows the library version so you can move ahead; come
back to your own afterwards. Use NumPy, SciPy sparse matrices and the provided helpers, not OpenCV's matcher,
homography or triangulation functions.

**Conventions.** Photos are numbered `0` to `N - 1`. A pair is a row `(i, j)` with `i < j`; a list of pairs has
shape `(M, 2)`. A verified pair is weighted by its number of agreeing matches. The landmark labels are for
evaluation only; your functions never use them.

In [ ]:
import collections
import itertools
import os
import re

import numpy as np
import scipy.sparse as sp

TOOLBOX_VERSION = "63efbeb30581"  # the toolbox this notebook was made with (filled in when it is built)
found = re.search(r'^TOOLBOX_VERSION = "(\w+)"', open(os.path.join("toolbox", "__init__.py")).read(), re.M)
if found is None or found.group(1) != TOOLBOX_VERSION:
    raise RuntimeError(
        "this notebook and your toolbox folder come from different releases. Delete the toolbox folder and "
        "rome_from_folder.py, and run the setup cell again (restart the kernel first if you had already imported "
        "the toolbox). If you still see this message after that, get the current notebook from your instructor (or "
        "the course repository) and copy your functions into it.")

import toolbox
from toolbox import *  # noqa: F401,F403  (every name in the table above, and ref)

print(f"toolbox ready: {len(toolbox.__all__)} tools")

## A folder full of Rome

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_castel_sant_angelo.jpg" alt="Ponte and Castel Sant&#x27;Angelo over the Tiber (Piranesi, 1754)" title="Ponte and Castel Sant&#x27;Angelo over the Tiber (Piranesi, 1754)" width="300" align="right" style="margin: 4px 0 10px 18px">

Marco scrolls through the first page of thumbnails.

**Marco:** "Someone's holiday photos got mixed in here. There's a lighthouse."

**Elena:** "Probably more than one holiday. Let's see what we can recover before tomorrow."

In [ ]:
rome = load_mini_rome(ROME_DIR)
N = len(rome["images"])
labels = rome["labels"]  # hidden: which landmark each photo shows, -1 for a stray
show_crate(rome, n=24, seed=0)

## 1. Which photos should we compare?

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_pantheon.jpg" alt="The portico and dome of the Pantheon (Piranesi, 1761)" title="The portico and dome of the Pantheon (Piranesi, 1761)" width="300" align="right" style="margin: 4px 0 10px 18px">

Marco opens two Trevi photos: the whole fountain, and a small part of its facade.

**Marco:** "These belong together. We could just compare each photo with all the others."

**Elena:** "How many comparisons would that be?"

$N$ photos make $N(N-1)/2$ pairs. The full collection's 8,987 photos give **40,378,591 pairs**: about 22 hours at
two milliseconds a check (an illustration; the notebook measures the real cost later). Most pairs show different
places, so we give each photo a quick shortlist of likely matches and check only those properly.

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/retrieval_vs_all_pairs.png" alt="TAV. II: checking every pair against checking only the top k of each photo" title="TAV. II: checking every pair against checking only the top k of each photo" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. II: checking every pair against checking only the top k of each photo</i></sub></p>

In [ ]:
n_pairs = N * (N - 1) // 2
print(f"{N} photos, {n_pairs:,} pairs")
print(f"at 2 ms per pair, checking every pair takes {n_pairs * 0.002 / 3600:.1f} hours")

The original project was far larger: about 150,000 Flickr photos, 496 cores, about 21 hours (13 h matching, 1 h
skeletal sets, 7 h SfM), and 8.8 million pairs checked instead of more than 11 billion.

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/rome_scale.png" alt="TAV. III: Building Rome in a Day against this notebook" title="TAV. III: Building Rome in a Day against this notebook" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. III: Building Rome in a Day against this notebook</i></sub></p>

**Further reading:** the [Building Rome in a Day project page](https://grail.cs.washington.edu/rome/) and its
[CACM article](https://doi.org/10.1145/2001269.2001293) (2011);
[Photo Tourism](https://phototour.cs.washington.edu/) (Snavely, Seitz, Szeliski 2006), the system it scaled up.

### Describing a photo with visual words

**Marco:** "How do we search a photo if it doesn't have a caption?"

**Elena:** "Look at its small details: corners, edges, bits of stonework. We can describe those with numbers."

A **keypoint** is an image location with a distinctive local pattern; its **descriptor** is a list of numbers
describing it. Here these are SIFT descriptors ([Lowe 2004](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf)), 128
numbers summarising the edge directions around the keypoint. Similar descriptors share an integer ID, a **visual
word**: a visual pattern, not a name like "fountain".

With 16,384 words, counting the words in a photo gives a histogram that is far cheaper to compare than the image. It
records which patterns occur, not where. The words of photo `i` are `words[offsets[i]:offsets[i + 1]]`; below,
features with the same word share a colour.

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/visual_words.png" alt="TAV. IV: features snapped to visual words; a photo becomes a weighted histogram" title="TAV. IV: features snapped to visual words; a photo becomes a weighted histogram" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. IV: features snapped to visual words; a photo becomes a weighted histogram</i></sub></p>

In [ ]:
offsets, words, n_words = rome["offsets"], rome["words"], len(rome["vocab"])
i = int(np.flatnonzero(labels >= 0)[0])  # the first photo of a landmark
j = int(np.flatnonzero(labels == labels[i])[1])  # another photo of the same landmark
print(f"photo {i}: {offsets[i + 1] - offsets[i]} keypoints, first words {words[offsets[i]:offsets[i] + 8]}")
show_words(rome, i, j)

### Task 1: `tfidf_matrix`

Marco finds two photos with similar-looking windows. They show different buildings.

**Marco:** "They share plenty of patterns, but it's the wrong place."

**Elena:** "Some patterns occur everywhere. A shared window corner tells us less than an unusual piece of carving."

**Tf-idf** weights them in two parts: **term frequency** (tf) counts a word in one photo; **inverse document
frequency** (idf) lowers the weight of words found in many photos. With $c_{iw}$ the count of word $w$ in photo $i$
and $n_w$ the number of photos containing it, each nonempty row is scaled to length 1:

$$T_{iw} \propto c_{iw}\left(\ln\frac{N}{n_w}+1\right), \qquad \|T_{i,:}\|_2=1.$$

**Your task:** write `tfidf_matrix(offsets, words, n_words)` and return a SciPy CSR sparse matrix, one row per photo
and one column per word (each photo uses only a small part of the vocabulary). Common words keep a weight; they are
not removed. The docstring gives the inputs, the output and the edge cases. Library version:
`sklearn.feature_extraction.text.TfidfTransformer(smooth_idf=False, norm="l2")`.

In [ ]:
def tfidf_matrix(offsets, words, n_words):
    """Every photo becomes one row of word weights: frequent words in this photo count more, common words count less.
    Re-implements sklearn.feature_extraction.text.TfidfTransformer(smooth_idf=False, norm="l2") on the word counts.

    Input:  offsets (N+1,) ints, the words of image i are words[offsets[i]:offsets[i+1]];
            words (P,) visual word ids in [0, n_words); n_words, the vocabulary size.
            offsets and words may be any array-like (lists too).
    Output: T, a scipy.sparse CSR matrix (N, n_words) float64: T[i, w] = tf * idf, every nonempty row has length 1
            (a photo with no words gives a row of zeros).
            tf = how often word w occurs in image i, idf = ln(N / n_w) + 1, n_w = number of images that contain w.
    """
    # YOUR CODE HERE
    raise NotImplementedError("tfidf_matrix")

In [ ]:
test_tfidf_matrix()

### Reading the similarity plot

For rows of length 1, the dot product is the [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity):
high when two photos share similar weighted patterns, 0 when they share no words. $T T^\top$ holds it for every
pair. The plot sorts a sample by landmark label: dark blocks on the diagonal mean photos of one landmark have
similar descriptions (the right panel averages per pair of landmarks). The search itself never sees the labels.

In [ ]:
T = tfidf_matrix(offsets, words, n_words)
print(f"T: {T.shape[0]} x {T.shape[1]}, {T.nnz:,} non zeros ({T.data.nbytes / 1e6:.0f} MB)")
plot_similarity_matrix(T, labels, n=600, seed=0, rome=rome)

### Task 2: `top_k_neighbours`

**Marco:** "So we keep the best few matches for each photo?"

**Elena:** "Yes. For now we just need a useful shortlist. We'll check the geometry next."

That is about $N k$ entries instead of every pair; the toolbox removes pairs listed from both sides. The Rome team
kept $k = 10$.

**Your task:** write `top_k_neighbours(T, k)`: for every photo, the indices and scores of its `k` most similar other
photos, best first, never itself. Compute the `N` by `N` similarity a block of rows at a time and keep the best `k`
of each row before moving on. The docstring has the shapes. Library version:
`sklearn.neighbors.NearestNeighbors(metric="cosine", algorithm="brute")`.

In [ ]:
def top_k_neighbours(T, k):
    """For every photo, the k photos that look most alike (cosine similarity of their word rows), best first.
    Re-implements sklearn.neighbors.NearestNeighbors(metric="cosine", algorithm="brute") with k + 1 neighbours, the
    photo itself dropped, score = 1 - distance.

    Input:  T (N, W) sparse, rows of length 1 (from tfidf_matrix), so cosine similarity is simply T T^T; k < N.
    Output: (idx (N,k) int64, score (N,k) float64): idx[i] are the k most similar other images, score[i] their
            similarities, in decreasing order. An image is never its own neighbour.
    """
    # YOUR CODE HERE
    raise NotImplementedError("top_k_neighbours")

In [ ]:
test_top_k_neighbours()

### Did the search find useful neighbours?

Green borders mark neighbours with the query's landmark label, red borders the others. A search always returns
neighbours, even for a photo from another city: a high rank is a reason to check a pair, not proof that it belongs
together. The precision plot shows the fraction of correct neighbours at each rank, averaged over Rome queries.
Compare the first few results with longer shortlists.

In [ ]:
TOP_K = 10
idx, score = top_k_neighbours(T, k=20)
show_neighbours(rome, idx, score, queries=[0, 1, 2, 3, 4], k=8)
plot_retrieval_precision(retrieval_precision(idx, labels))

**Think about it:** What does a wrong shortlist entry cost us at this stage? What might happen if a useful pair
never reaches the shortlist?

**Further reading:** [Video Google](https://www.robots.ox.ac.uk/~vgg/research/vgoogle/) (Sivic and Zisserman 2003),
where visual words and tf-idf for images began; the [vocabulary tree](https://doi.org/10.1109/CVPR.2006.264) (Nister
and Stewenius 2006), the far larger vocabulary the Rome team used.

## 2. Do these photos show the same place?

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_forum.jpg" alt="The Roman Forum as Campo Vaccino (Piranesi, 1772)" title="The Roman Forum as Campo Vaccino (Piranesi, 1772)" width="300" align="right" style="margin: 4px 0 10px 18px">

The search puts two photos of stone arches side by side. Marco zooms in.

**Marco:** "I can see why the search picked these. I'm still not sure they're the same arches."

**Elena:** "Let's match individual points and see whether they fit one consistent pair of camera views."

In [ ]:
pairs = neighbour_pairs(idx[:, :TOP_K])
print(f"{len(pairs):,} proposed pairs instead of {n_pairs:,}")

### Task 3: `match_descriptors`

Marco tries to pick out the same details in two Trevi photos.

**Marco:** "I can match this statue. The windows are harder. Three of them look almost identical."

**Elena:** "Then we shouldn't force a choice. A match is useful when one candidate is clearly better than the
others."

The nearest descriptor in the other photo (Euclidean distance) is the first candidate, but nearest is not always
convincing. Two checks:

- **Ratio test** ([Lowe 2004](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf), section 7.1): with $d_1$, $d_2$ the
  distances to the nearest and second nearest, keep the match only if $d_1 < r\, d_2$. With $r = 0.8$, distances 4
  and 10 pass; 9 and 10 are too ambiguous.
- **Mutual check:** if `i` chooses `j`, then `j` must choose `i` when we search the other way.

**Your task:** write `match_descriptors(d1, d2, ratio=0.8)` and return the kept pairs `(i, j)` (row `i` of `d1`, row
`j` of `d2`) as an `(M, 2)` array sorted by `i`. The ratio test runs from `d1` to `d2` only; the mutual check uses
the reverse nearest neighbour. Compute the distances a block of rows at a time with one matrix product, using
$\|\mathbf{a}-\mathbf{b}\|^2=\|\mathbf{a}\|^2+\|\mathbf{b}\|^2-2\,\mathbf{a}\cdot\mathbf{b}$. The docstring lists
the edge cases. Library version: OpenCV's `BFMatcher` (`knnMatch` with `k = 2`, plus a reverse search).

In [ ]:
def match_descriptors(d1, d2, ratio=0.8):
    """Pair up the keypoints of two photos by their descriptors, keeping only the pairings that are clear and agreed.
    Re-implements cv2.BFMatcher(cv2.NORM_L2): knnMatch(d1, d2, k=2) with Lowe's ratio test, plus match(d2, d1) for
    the mutual check.

    Input:  d1 (N1, D), d2 (N2, D) descriptors, one row per keypoint (float32, float16 or uint8, or any array-like;
            D = 128 for SIFT); ratio, Lowe's ratio (0.8).
    Output: (M,2) int64 pairs (i, j): row i of d1 goes with row j of d2, sorted by i. Pair (i, j) is kept when
              * j is the nearest row of d2 to d1[i] (Euclidean distance; ties go to the smallest index),
              * that distance is below ratio times the distance to the second nearest row of d2 (the ratio test),
              * and i is also the nearest row of d1 to d2[j] (mutual check; ties to the smallest index).
            Fewer than 2 rows on either side gives an empty (0, 2) array.
    """
    # YOUR CODE HERE
    raise NotImplementedError("match_descriptors")

In [ ]:
test_match_descriptors()

### Checking the geometry

**Marco:** "These matches look much better. Can we trust them now?"

**Elena:** "Not yet. Similar patterns can pass both tests. We check where the points sit in the images."

True matches agree with one camera geometry; look-alikes usually do not. The 3 by 3 **fundamental matrix** `F`
describes this [epipolar geometry](https://en.wikipedia.org/wiki/Epipolar_geometry): a point in one image must lie
near its **epipolar line** in the other. With pixels written as $(u, v, 1)$, a correct match satisfies

$$\tilde{\mathbf{x}}_2^\top F\tilde{\mathbf{x}}_1 \approx 0.$$

**Marco:** "What if some of the matches are wrong?"

**Elena:** "We expect that. We fit models to small random samples and keep one that many matches agree with."

That is **RANSAC**: fit `F` to eight random matches, count the matches that fit within a tolerance (the
**inliers**), repeat, keep the best. A model with many inliers is unlikely to rest on a few bad matches.

`show_match_filters` matches one Trevi pair three ways (nearest neighbour; plus the ratio test; plus the mutual
check, your matcher) and checks each with RANSAC: green lines are inliers, red lines fail. Compare the counts and
the fraction that fit.

In [ ]:
trevi = photos_of(rome, "Trevi")  # the hidden labels, only to pick a good example
i = int(next(q for q in trevi if labels[idx[q, 0]] == labels[q]))  # a Trevi photo whose best neighbour is Trevi too
j = int(idx[i, 0])
cost = show_match_filters(rome, i, j)

### Choosing where to spend the time

**Marco:** "Let's run that on every pair in the shortlist."

**Elena:** "Check the timing first. That pair was quick, but we have thousands left."

The next cell estimates full SIFT on every photo plus your matching on every proposed pair.

In [ ]:
minutes = (N * cost["sift_s"] + len(pairs) * cost["match_s"]) / 60
print(f"SIFT on {N:,} photos and your match_descriptors on {len(pairs):,} proposed pairs: about {minutes:.0f} min")

Too slow for a first pass. The provided `verify_pairs` takes a shortcut: words that occur exactly once in each photo
propose point matches, then OpenCV's RANSAC estimates `F`. Pairs with enough inliers are accepted, with their inlier
counts as edge weights. Below, the same pair matched by words. Your careful `match_descriptors` returns in Chapter
4, where every match becomes a 3D point.

In [ ]:
show_word_matches(rome, i, j)

Now all the proposals (a minute or two). An accepted pair is good evidence of overlap, though repeated patterns and
difficult geometry can still fool it.

In [ ]:
t0 = time.time()
n_inl, ok = verify_pairs(pairs, rome)
ms_per_pair = 1000 * (time.time() - t0) / len(pairs)
edges_retrieval, weights_retrieval = pairs[ok], n_inl[ok]  # kept: query expansion starts from these
print(f"{ok.sum():,} of {len(pairs):,} proposals are real ({ms_per_pair:.1f} ms per pair)")
print(f"checking all {n_pairs:,} pairs would have taken {n_pairs * ms_per_pair / 3.6e6:.1f} hours")

### Task 4: `query_expansion`

Two Trevi photos, a wide view and a close-up, were never compared, but both matched a third.

**Marco:** "This middle photo overlaps with both. Should we compare the other two?"

**Elena:** "Yes. They may share something the first search missed."

If `i` matches `j` and `j` matches `k`, then `(i, k)` is worth checking: a candidate, not a match, since the two may
show different parts of what `j` sees. This is **query expansion**. With photos as nodes and verified pairs as
edges, a path of two edges gives a candidate; with adjacency matrix $A$, $(A^2)_{ik}$ counts the neighbours `i` and
`k` share.

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/query_expansion.png" alt="TAV. V: friends of friends on a ring of Trevi photos" title="TAV. V: friends of friends on a ring of Trevi photos" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. V: friends of friends on a ring of Trevi photos</i></sub></p>

**Your task:** write `query_expansion(edges, tested)` and return the new pairs `(i, k)`, `i < k`, that share a
neighbour, without self-pairs, repeats or pairs already in `tested`. The docstring gives the order and the empty
case. Library version: a SciPy sparse adjacency matrix times itself.

In [ ]:
def query_expansion(edges, tested):
    """Friends of friends: if photo j matched both i and k, then i and k probably show the same place too.

    Re-implements A @ A with scipy.sparse on the adjacency matrix A of the verified pairs: (A @ A)[i, k] > 0 exactly
    when i and k have a common neighbour.

    Input:  edges (E,2) ints (any array-like), verified pairs (i, j), in either order;
            tested, a set of (i, j) tuples with i < j: pairs already checked.
    Output: (M,2) int64 array of new candidate pairs (i, k), i < k, sorted (by i, then k), no repeats,
            not in tested, never (i, i). An empty result has shape (0, 2).
    """
    # YOUR CODE HERE
    raise NotImplementedError("query_expansion")

In [ ]:
test_query_expansion()

Two rounds, each verified with `verify_pairs`. Large landmarks propose many pairs, so each round checks at most
`MAX_NEW`, picked at random. The Rome team ran four rounds; after that, few proposals were real.

In [ ]:
ROUNDS, MAX_NEW = 2, 20000
rng = np.random.default_rng(0)
tested = set(map(tuple, pairs.tolist()))
edges, weights = edges_retrieval.copy(), weights_retrieval.copy()  # from the retrieval pairs, so a rerun is the same
for r in range(ROUNDS):
    new = query_expansion(edges, tested)
    if len(new) > MAX_NEW:
        new = new[np.sort(rng.choice(len(new), MAX_NEW, replace=False))]
    n_inl_new, ok_new = verify_pairs(new, rome)
    tested |= set(map(tuple, new.tolist()))
    edges, weights = np.concatenate([edges, new[ok_new]]), np.concatenate([weights, n_inl_new[ok_new]])
    print(f"round {r + 1}: {len(new):,} proposed, {ok_new.sum():,} real, {len(edges):,} edges in total")

The **match graph**: one dot per photo (coloured by landmark label), one line per verified pair, before and after
expansion. New edges mean more pairs passed verification; the photos have not changed.

In [ ]:
plot_match_graph(N, edges_retrieval, labels, kind="labels", rome=rome,
                 title=f"after retrieval: {len(edges_retrieval):,} edges")
plot_match_graph(N, edges, labels, kind="labels", rome=rome,
                 title=f"after query expansion: {len(edges):,} edges")

**Think about it:** What fraction of the new candidates pass? How does that compare with the original shortlists?
Can these new edges join previously separate connected components?

**Further reading:** [RANSAC](https://doi.org/10.1145/358669.358692) (Fischler and Bolles 1981), whose title ends in
"automated cartography"; [Lowe's SIFT paper](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf), section 7.1, with the
plot that justifies 0.8; [Szeliski's free book](https://szeliski.org/Book/) on feature matching and epipolar lines.

## 3. Which photos belong together?

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/map_nolli_1748.jpg" alt="Nolli&#x27;s 1748 plan of Rome inside the Aurelian walls" title="Nolli&#x27;s 1748 plan of Rome inside the Aurelian walls" width="300" align="right" style="margin: 4px 0 10px 18px">

The graph has several large groups and many small ones. Marco opens a large one.

**Marco:** "Mostly the Pantheon. But there are more Pantheon photos in another group."

**Elena:** "The graph only knows the matches we've found, and inside and outside views barely overlap. Nolli's plan
leaves streets, squares and even the inside of the Pantheon white: the city as space you can walk through. Our
groups are the same, walked photo to photo along verified matches."

A **connected component** is a group of photos linked by paths of verified matches; two of its photos need not match
directly. Components often match landmarks, but check: a landmark can split, neighbours can join where photos
overlap, and one bad edge can join unrelated groups.

### The provided `UnionFind`

`UnionFind` (a [disjoint-set structure](https://en.wikipedia.org/wiki/Disjoint-set_data_structure)) starts with each
photo in its own group and merges groups as edges connect them.

- `find(i)` returns the root identifying photo `i`'s group.
- `union(i, j)` merges their groups and returns `True` if they were separate.
- Path compression speeds up repeated lookups; smaller groups are attached to larger ones.

In [ ]:
uf = UnionFind(5)
print(uf.union(0, 1), uf.union(1, 2), uf.union(0, 2))  # True True False: the last one was already joined
print([uf.find(i) for i in range(5)])

### Task 5: `connected_components`

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/connected_components.png" alt="TAV. VI: the match graph split into connected components, with lone photos" title="TAV. VI: the match graph split into connected components, with lone photos" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. VI: the match graph split into connected components, with lone photos</i></sub></p>

**Your task:** write `connected_components(n, edges)`: `union` the ends of every edge, then label each photo by its
component, `0` for the largest, `1` for the next, and so on. An isolated photo is a component of size 1. The
docstring gives the tie rule. Library version: `scipy.sparse.csgraph.connected_components`, relabelled by size.

The Rome team used the same routine twice: for landmarks (photos joined by verified pairs) and for tracks (keypoints
joined by matches, Chapter 4).

In [ ]:
def connected_components(n, edges):
    """Split the match graph into islands: two photos are in one island when a chain of verified pairs links them.
    Re-implements scipy.sparse.csgraph.connected_components, with the labels renumbered by size.

    Input:  n, the number of nodes (images); edges (E,2) ints in [0, n) (any array-like).
    Output: labels (n,) int64. Component 0 is the largest, then 1, 2, ... by decreasing size; components of equal size
            are ordered by their smallest member. A node with no edge is a component of size 1.
    """
    # YOUR CODE HERE
    raise NotImplementedError("connected_components")

In [ ]:
test_connected_components()

### Checking the groups

The graph coloured by your components, then galleries of the largest. The purity table uses the labels to measure
what fraction of each component shows its most common landmark. Look at coverage too: a small all-Trevi group is
perfectly pure while missing most of the Trevi photos.

In [ ]:
comp = connected_components(N, edges)
sizes = np.bincount(comp)
print(f"{len(sizes)} components; the largest: {sizes[:10].tolist()}; {np.sum(sizes == 1)} photos left alone")
plot_match_graph(N, edges, comp, kind="components", title="components")
show_components(rome, comp, n_components=9, n_photos=8)
component_purity(comp, labels, rome=rome)

**Think about it:** Do St Peter's Square and Basilica end up together? Are all isolated photos from other cities? If
a landmark splits into several groups, what do their galleries suggest about the missing overlap?

**Further reading:** [disjoint sets](https://en.wikipedia.org/wiki/Disjoint-set_data_structure), and why path
compression makes them nearly constant time; the [CACM article](https://doi.org/10.1145/2001269.2001293) on Rome.

## 4. Building the Trevi Fountain

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_trevi.jpg" alt="The Trevi fountain in perspective (Piranesi, 1773)" title="The Trevi fountain in perspective (Piranesi, 1773)" width="300" align="right" style="margin: 4px 0 10px 18px">

Elena selects the group with most of the Trevi photos.

**Marco:** "We've found the photos. Now we need to work out where all those cameras were."

**Elena:** "We'll build an initial model from a smaller set first, then use its 3D points to place the rest."

Incremental SfM adds cameras one by one and keeps refining the model, which gets expensive. The **skeletal set**
approach reconstructs a smaller set of photos that holds the shape together, then registers the others against its
points.

In [ ]:
c = component_of_landmark(comp, rome, "Trevi")  # the component that holds most Trevi photos
members, e_c, w_c = component_graph(comp, c, edges, weights)  # local ids 0 .. len(members) - 1
print(f"component {c}: {len(members)} photos, {len(e_c):,} verified edges")

### The skeleton: a maximum spanning tree

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/maximum_spanning_tree.png" alt="TAV. VII: the skeleton, a maximum spanning tree found by Kruskal" title="TAV. VII: the skeleton, a maximum spanning tree found by Kruskal" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. VII: the skeleton, a maximum spanning tree found by Kruskal</i></sub></p>

**Marco:** "How do we choose the links to start from?"

**Elena:** "The strongest matches, with as few edges as it takes to keep everything connected."

A **spanning tree** connects every node of a connected graph without cycles: `n - 1` edges for `n` nodes. A
**maximum spanning tree** has the largest total weight, here the inlier counts.
[Kruskal's algorithm](https://en.wikipedia.org/wiki/Kruskal%27s_algorithm) visits edges from heaviest to lightest
and keeps one only if it joins two separate groups, which the provided `UnionFind` can tell (as in Task 5).

The toolbox's `maximum_spanning_tree(n, edges, weights)` runs Kruskal's algorithm and returns a boolean mask marking
the tree's edges. Several components give one tree each (a spanning forest); self-loops are never kept. It agrees
with `scipy.sparse.csgraph.minimum_spanning_tree` on the negated weights; `help(maximum_spanning_tree)` describes it.

In the plot below, thick lines are the tree, thin grey lines the rest of the graph. The toolbox then peels off outer
leaves, keeping a minimum number of skeleton photos. The tree only organises this selection: other verified matches
still contribute tracks, and cycles carry useful geometric constraints. Dropping them here is a selection step, not a
claim that they hold no information.

In [ ]:
keep = maximum_spanning_tree(len(members), e_c, w_c)
print(f"tree: {keep.sum()} of {len(e_c):,} edges, weight {w_c[keep].sum():,} of {w_c.sum():,}")
plot_spanning_tree(len(members), e_c, keep, title="Trevi: the maximum spanning tree")

### From matching pixels to 3D points

`landmark_scene` recomputes SIFT for the group's photos and runs **your `match_descriptors`** on selected strong
verified pairs. RANSAC filters the matches, then `UnionFind` links observations across images into **tracks**: image
points believed to show one physical point. A minute or two.

In [ ]:
scene = landmark_scene(rome, members, e_c, w_c)

**Marco:** "We have the same corner in two photos. How does that give us its depth?"

**Elena:** "Once we know the cameras, each pixel is a viewing ray from its camera, and the point is where the rays
meet. Surveyors fixed church towers the same way: sight the tower from two known stations, and it stands where the
lines cross."

This is [triangulation](https://en.wikipedia.org/wiki/Triangulation_%28surveying%29); noisy rays rarely meet
exactly, so we take the best fitting point. The starting pair needs overlap and a real step between the cameras, a
**baseline**: a surveyor who turns without walking to a second station measures angles but never distances, and so
does a camera that only rotates. Tiny baselines make depth unreliable too.

### Task 6: `homography_dlt`

Marco sorts candidate pairs by shared tracks. The top pair looks like two copies of one photo.

**Marco:** "This pair has loads of matches. It should be a good place to start."

**Elena:** "Maybe. But if the photographer barely moved, those matches won't tell us much about depth."

A camera rotating about its centre gives images related by a
[homography](https://en.wikipedia.org/wiki/Homography_%28computer_vision%29), a 3 by 3 matrix $H$ with
$\tilde{\mathbf{x}}_2 \sim H \tilde{\mathbf{x}}_1$ (equal up to scale) whatever the point's depth; so does a flat
surface seen from two positions. Many matches can fit one $H$ and still make a poor start, so we fit $H$ to test for
it.

**Marco:** "So this matrix moves points from one image to the other?"

**Elena:** "Yes, for the cases it describes. It gives no depth for each point."

$H$ has nine entries but only eight count (scaling them all changes nothing), and each match gives two equations:
four matches in a suitable arrangement fix it. The **direct linear transform** (DLT) stacks the equations as
$A\mathbf{h} = 0$ and takes the unit $\mathbf{h}$ that makes $\|A\mathbf{h}\|$ smallest, the last right singular
vector of $A$. As in `eight_point`, first centre and rescale the points with `hartley_normalize`, or pixels in the
hundreds leave $A$ badly balanced.

**Your task:** write `homography_dlt(x1, x2)` for two `(N, 2)` arrays of matching pixels and return `H`, mapping
`[u, v, 1]` in the first image to the match in the second (after dividing by the third coordinate). Normalise,
build `A` from two equations per match, solve by SVD, undo the normalisation (`H = inv(T2) @ H_normalised @ T1`) and
scale so `H[2, 2] = 1`. Raise `ValueError` when the points do not fix one `H`; the docstring lists the cases. Library
version: `cv2.findHomography(x1, x2, 0)` (all points, no RANSAC).

In [ ]:
def homography_dlt(x1, x2):
    """The homography H with x2 ~ H x1 (pixels with a 1 appended), by the normalised DLT (direct linear transform).
    Re-implements cv2.findHomography(x1, x2, 0) (all points, no RANSAC).

    Input:  x1, x2 (N,2) pixels (any array-like), row k of x1 matches row k of x2, N >= 4.
    Output: H (3,3) float64, scaled so that H[2,2] = 1.
            Raises ValueError when the matches fix no proper homography:
              * fewer than 4 matches;
              * more than one H fits them (the DLT system has rank under 8): this happens when the points are degenerate
                in both photos, e.g. all on one line, 3 of 4 on one line, or all but one on one line;
              * the only H that fits is singular (it flattens the plane onto a line or a point): this happens when
                the points of one photo are degenerate and those of the other are not, e.g. x1 is a proper
                quadrilateral but 3 of the 4 points of x2 lie on one line, or x2 holds the same point twice.
    """
    # YOUR CODE HERE
    raise NotImplementedError("homography_dlt")

In [ ]:
test_homography_dlt()

### Task 7: `homography_ratio`

**Marco:** "How do we tell whether the homography explains the pair well enough to be a problem?"

**Elena:** "Count its inliers, and compare with the fundamental matrix's inliers on the same matches."

$$\text{ratio}=\frac{\text{number of }H\text{ inliers}}{\max(\text{number of }F\text{ inliers},1)}$$

Near 1, one homography explains about as much as `F`: pure rotation, a nearly flat scene, or too little movement to
reveal depth. A lower ratio suggests **parallax**, points at different depths shifting differently between camera
positions. COLMAP runs the same test to pick its first pair.

It is a screening test, not a measurement of camera separation. A low ratio still needs reliable matches and a
passing geometry check, and a high ratio from a flat wall does not prove the camera stood still: planar matches
leave `F` ambiguous even when the cameras moved.

**Your task:** write `homography_ratio(x1, x2, thresh=2.0, iters=500, seed=0)` and return
`(ratio, H, inliers_H, inliers_F)`. Estimate `H` by RANSAC with your `homography_dlt`: fit four random matches (skip
samples that raise `ValueError`), count the matches whose **transfer error** (pixel distance from $\mathbf{x}_2$ to
the mapped $H\tilde{\mathbf{x}}_1$) is below `thresh`, keep the best model and refit on its inliers. The `F` inliers
come from the provided `ransac_fundamental`. The docstring covers seeding and the case of no valid `H`. Library
version: OpenCV's `findHomography` and `findFundamentalMat` with RANSAC.

In [ ]:
def homography_ratio(x1, x2, thresh=2.0, iters=500, seed=0):
    """Is there a real step (baseline) between the two cameras? Fit one homography H and one fundamental matrix F to
    the same matches and compare how many each explains. Re-implements cv2.findHomography(x1, x2, cv2.RANSAC) next to
    cv2.findFundamentalMat(x1, x2, cv2.FM_RANSAC).

    Input:  x1, x2 (N,2) matched pixels, N >= 8 (F needs 8); thresh in px; iters RANSAC rounds; seed for the samples
            (draw them with np.random.default_rng(seed), so the same seed gives the same result).
    Output: (ratio float, H (3,3) or None, inliers_H (N,) bool, inliers_F (N,) bool),
            ratio = H inliers / max(F inliers, 1), inliers_F from the provided ransac_fundamental(x1, x2,
            thresh=thresh, iters=iters, seed=seed). If no sample gives an H (all degenerate), H is None and
            inliers_H is all False.
            A ratio near 1 means one homography explains the pair: the camera only turned (or the scene is flat), so
            the pair gives no reliable depth. A low ratio means real parallax.
    """
    # YOUR CODE HERE
    raise NotImplementedError("homography_ratio")

In [ ]:
test_homography_ratio()

### Choosing the first pair

Your test on 60 Trevi pairs: the 30 sharing the most tracks and 30 spread over the next 570 (about 15 seconds). A has
the highest ratio, B the lowest. Red lines are matches explained by `H`; green lines fit `F` but not `H`.

In [ ]:
ranked = strongest_pairs(scene, n=600)  # most shared tracks first
results = []
for a, b in np.concatenate([ranked[:30], ranked[30::19][:30]]):
    x1, x2 = pair_matches(scene, a, b)
    ratio, H, inl_H, inl_F = homography_ratio(x1, x2)
    results.append((a, b, ratio, inl_H.sum(), inl_F.sum()))
show_homography_test(scene, results)

**Marco:** "Pair A has plenty of matches, but most of them fit one homography."

**Elena:** "Look at B. The facade is flat and may still fit a homography, but the basin and rocks in front shift
differently. That's the depth we're looking for."

`run_sfm_skeleton` tests the 60 skeleton pairs sharing the most tracks with your `homography_ratio` (at a looser 4 px
threshold, so its ratios differ a little from the test above), ranks them by shared tracks times
$\max(0, 1 - \text{ratio})$, tries the best 8 and keeps the one with the most good points (in front of both
cameras, rays meeting at more than 2 degrees; at least 30 needed). It then adds the other skeleton cameras one at a
time, with bundle adjustment every few cameras and at the end.

**Think about it:** Why might two nearly identical photos have more matches than a pair that is better for
recovering depth? Why does observing only a flat wall make this test harder to interpret?

### Task 8: `triangulate_points`

**Marco:** "We have a good first pair. How do its matches become 3D points?"

**Elena:** "Like the surveyor's two sightings. Each camera turns a pixel into a ray; the point sits where the two
rays meet."

This is **triangulation**, and SfM cannot do without it: the cameras are found from matches, but the model's points
only exist once each match is lifted into 3D. Every later step builds on them: PnP places new photos against the
points, bundle adjustment refines points and cameras together, and the dense cloud is triangulated pixel by pixel.

One photo is not enough. A pixel says in which direction the point lies from the camera, not how far away it is:
every point on the ray lands on the same pixel. A second photo from somewhere else sees the same point along another
ray, and two rays that are not parallel cross in one place.

A camera is summed up by its **projection matrix** $P = K[R \mid \mathbf{t}]$, a $3 \times 4$ matrix
(`projection_matrix(K, R, t)` builds it). It sends a 3D point, written in homogeneous coordinates
$\mathbf{X} = (X, Y, Z, 1)$, to the pixel it is seen at: $P\mathbf{X} = (a, b, c)$ and the pixel is
$(u, v) = (a / c,\; b / c)$. Writing $P^{(1)}, P^{(2)}, P^{(3)}$ for the rows of $P$, that says
$u = P^{(1)}\mathbf{X} / P^{(3)}\mathbf{X}$ and $v = P^{(2)}\mathbf{X} / P^{(3)}\mathbf{X}$; multiply out and each
pixel gives two equations that are linear in $\mathbf{X}$:

$$u\,P^{(3)}\mathbf{X} - P^{(1)}\mathbf{X} = 0, \qquad v\,P^{(3)}\mathbf{X} - P^{(2)}\mathbf{X} = 0.$$

Two photos give four equations, the **direct linear transform** (DLT) again, as for the homography:

$$
A=
\begin{bmatrix}
u_1P_1^{(3)}-P_1^{(1)}\\
v_1P_1^{(3)}-P_1^{(2)}\\
u_2P_2^{(3)}-P_2^{(1)}\\
v_2P_2^{(3)}-P_2^{(2)}
\end{bmatrix},
\qquad A\mathbf{X} = 0.
$$

$\mathbf{X}$ has four entries but only three count (scaling it changes nothing), and four equations are one too
many: with noisy pixels the rays miss each other and no $\mathbf{X}$ fits exactly. So, as in Task 6, take the unit
$\mathbf{X}$ that makes $\|A\mathbf{X}\|$ smallest, the last row of $V^\top$ in the SVD $A = U \Sigma V^\top$. Last,
turn the homogeneous answer $(X_1, X_2, X_3, X_4)$ into an ordinary 3D point by dividing by the fourth coordinate:
$(X_1 / X_4,\; X_2 / X_4,\; X_3 / X_4)$. A fourth coordinate near 0 means the rays are nearly parallel and the
point is very far away.

**Your task:** write `triangulate_points(P1, P2, pts1, pts2)` for two $3 \times 4$ projection matrices and two
`(N, 2)` arrays of matching pixels, and return the `(N, 3)` points: one $4 \times 4$ matrix $A$ and one SVD per
point, then the division. Library version: `cv2.triangulatePoints(P1, P2, pts1.T, pts2.T)`, which returns the
homogeneous points as columns.

In [ ]:
def triangulate_points(P1, P2, pts1, pts2):
    """Two views of the same points give their 3D positions: linear triangulation (DLT), one small SVD per point.
    Re-implements cv2.triangulatePoints(P1, P2, pts1.T, pts2.T), divided by its fourth row.

    Input:  P1, P2 (3,4) camera projection matrices, a pixel x ~ P [X, 1] (any array-like; any nonzero scale);
            pts1, pts2 (N,2) pixels (any array-like): row k of pts1 and row k of pts2 show the same 3D point.
    Output: points_3d (N,3) float64, the Euclidean coordinates of each point: the homogeneous solution of A X = 0,
            divided by its fourth coordinate.
    """
    # TODO: for each point, stack the four rows of A: u1 P1[2] - P1[0], v1 P1[2] - P1[1], and the same for camera 2
    # TODO: solve A X = 0 with np.linalg.svd(A): X is the last row of Vt (the unit vector that makes |A X| smallest)
    # TODO: divide X by its fourth coordinate and keep the first three
    # YOUR CODE HERE
    raise NotImplementedError("triangulate_points")

In [ ]:
test_triangulate_points()

### Experiment A: reprojection error

Back to Trevi. `triangulation_pair` goes through the pairs of the test above from the lowest ratio (the most parallax)
up and takes the first that passes every check: enough matches fit its fundamental matrix, a camera pose comes out of
them, and the toolbox's own triangulation puts nearly all the points in front of both cameras with rays meeting at a
median angle of at least 2 degrees. Pair B, the lowest ratio, is the first it tries; the checks may still pass it over
(for example when too many of its points land behind a camera), and its output says why. `pair_cameras` then estimates
the pair's two cameras from its matches alone, and your function lifts every match into 3D. Projecting each point back
into both photos gives the **reprojection error**: how many pixels the predicted position lands from the keypoint that
was observed.

In [ ]:
a, b = triangulation_pair(scene, results)  # the lowest ratio that passes every check
P1, P2, x1, x2 = pair_cameras(scene, a, b)  # x1, x2: the matches that fit the pair's fundamental matrix
X = triangulate_points(P1, P2, x1, x2)


def reproject(P, X):
    """3D points (N,3) back into a photo: P [X, 1], then divide by the third coordinate."""
    p = np.column_stack([X, np.ones(len(X))]) @ P.T
    return p[:, :2] / p[:, 2:]


x1_hat, x2_hat = reproject(P1, X), reproject(P2, X)
err = np.linalg.norm(np.concatenate([x1_hat - x1, x2_hat - x2]), axis=1)
Xh = np.column_stack([X, np.ones(len(X))])
in_front = (Xh @ P1[2] > 0) & (Xh @ P2[2] > 0)  # depth along each camera's axis (the third row of P = K [R | t])
print(f"photos {a} and {b}: {len(X):,} points; reprojection error median {np.median(err):.2f} px, "
      f"90% under {np.percentile(err, 90):.2f} px; {in_front.sum():,} in front of both cameras")
show_reprojection(scene, a, b, x1, x2, x1_hat, x2_hat)

Most points land well under a pixel from where they were seen: the two rays nearly meet, and the triangulated point
sits between them. (The red lines are drawn many times longer than the errors, so they can be seen.) The longer lines
belong to matches that only just fit the geometry, or that are slightly off. The DLT minimises $\|A\mathbf{X}\|$, a
convenient algebraic quantity rather than the pixel error itself; bundle adjustment, below, minimises the pixel error
over all photos at once.

**In front of both cameras.** A real point lies in front of every camera that sees it: its depth, the third
coordinate of $P\tilde{\mathbf{X}}$, is positive in both photos (the **cheirality** condition). The algebra above
does not know this: $A\mathbf{X} = 0$ is just as happy with a point behind a camera, since a ray extended backwards
also passes through the same pixels. That is why the essential matrix gives four candidate poses and `choose_pose`
keeps the one that puts the most points in front of both cameras, and why a point behind a camera usually means a
wrong match or rays so nearly parallel that noise flips their crossing to the other side. Here the count above
should be all or nearly all of them.

A small reprojection error does not prove the depth is right, though: a point slid along its ray projects to almost
the same pixel. How well depth is pinned down depends on the angle between the rays, which the next experiment
shows.

### Experiment B: the baseline

Here everything is known: synthetic points about 10 units away, camera 1 at the origin and camera 2 moved sideways
by the **baseline** $b$, every pixel disturbed by 1 px of noise. Your `triangulate_points` rebuilds the points for
baselines from 0.02 to 4 units, and the plot compares them with the truth.

In [ ]:
baselines, errors = baseline_experiment(noise=1.0)
for b_, e_ in zip(baselines[::3], errors[::3]):
    print(f"baseline {b_:5.2f}: median 3D error {e_:6.3f}")

The error falls in proportion to $1/b$: double the baseline, halve the error. With a short baseline the two rays are
almost parallel, so a pixel of noise slides their crossing a long way along the rays. For cameras side by side the
depth is $Z = f b / d$, with $d$ the **disparity** (how far the point shifts between the photos, in pixels), so a
disparity error $\delta d$ moves the depth by about $Z^2 \delta d / (f b)$: far points and short baselines are hit
hardest, as the top view shows. This is why `homography_ratio` steers away from pairs taken from one spot, and why
`run_sfm_skeleton` keeps only points whose rays meet at a large enough angle (2 degrees for the start pair, 1.5
degrees for the points added later).

**What linear triangulation cannot do.** It trusts its matches: a wrong or noisy match gives a confident wrong
point, and the DLT minimises an algebraic quantity, not the pixel error, so with noise its answer is good but not
the best (bundle adjustment fixes that with every photo at once). It needs a baseline, as above. And it struggles
with points at infinity: for a very far point (the sky, distant hills) the two rays are parallel, the fourth
coordinate of the solution is close to 0, and dividing by it gives huge, unstable coordinates even though the
point's direction is right. SfM keeps such points out of the model by their ray angle.

**Think about it:** Why does a large baseline not help without limit? (Think about what happens to the matches when
two photos see the scene from very different angles.)

### Refining the model

**Marco:** "What is bundle adjustment changing?"

**Elena:** "The camera poses and the 3D points, together, so the model predicts the measured image points better.
Surveyors adjust a whole network of sightings the same way, by
[least squares](https://en.wikipedia.org/wiki/Least_squares), invented around 1800 for geodesy and astronomy."

Project a point back into a photo: its distance from the measured keypoint is the **reprojection error**.
[Bundle adjustment](https://en.wikipedia.org/wiki/Bundle_adjustment) moves all cameras $P_i$ and points
$\mathbf{X}_p$ to reduce $\sum_{i, p} \|\mathbf{x}_{ip} - \pi(P_i, \mathbf{X}_p)\|^2$. Its cost grows with the
model, so running it often gets expensive.

In [ ]:
t0 = time.time()
rec = run_sfm_skeleton(scene, e_c[keep])
t_skeleton = time.time() - t0
plot_reconstruction(rec, scene, title=f"Trevi skeleton: {len(rec['cams'])} photos, {len(rec['pts']):,} points")

### Your triangulation inside the model

The model's points come from the toolbox's triangulation, then bundle adjustment refines them with every photo that
sees them. Take the two placed photos that share the most of those points among the pairs with a real baseline (rays
meeting at a median angle of at least 3 degrees: as Experiment B showed, near copies of one view triangulate
poorly), other than the start pair, whose points were first triangulated from exactly those two photos. Build their
projection matrices from the model's cameras and triangulate their pixels with your function: how close do two
photos come to the adjusted points? The pipeline itself does not change.

In [ ]:
m1, m2, P_m1, P_m2, x_m1, x_m2, X_model = model_pair(rec, scene)  # most shared points, a real baseline, not the start
X_yours = triangulate_points(P_m1, P_m2, x_m1, x_m2)
size = np.linalg.norm(np.percentile(X_model, 95, axis=0) - np.percentile(X_model, 5, axis=0))
gap = np.linalg.norm(X_yours - X_model, axis=1) / size
print(f"photos {m1} and {m2}: {len(X_model):,} shared points; yours are within {np.median(gap):.2%} of the "
      f"model's size (median), 90% within {np.percentile(gap, 90):.2%}")
compare_with_model(X_yours, X_model, title=f"photos {m1} and {m2}: your triangulation against the model's points")

Most of your points sit almost on the model's: with the model's cameras, two photos already pin a point down well.
The ones further off are where the two photos see the point at a narrow angle, or where the other photos pulled the
adjusted point away from what these two alone suggest: more rays, measured in pixels, give a better estimate.

### Placing the remaining cameras

**Marco:** "Do the remaining photos have to go through the same process?"

**Elena:** "No. A surveyor who doesn't know where they stand sights three or more landmarks already on the map and
works out their position. A photo that sees enough of our points can do the same."

Surveyors call it [resection](https://en.wikipedia.org/wiki/Position_resection_and_intersection); in vision it is
**PnP** (Perspective-n-Point): a camera's pose from known 3D points and their image positions. `register_rest` tries
every remaining photo against the skeleton's points, adds those with enough reliable matches in one batch,
triangulates more points and runs a final bundle adjustment. Photos without enough usable overlap stay out. Compare
the cameras placed and the time taken in the two stages.

In [ ]:
t0 = time.time()
rec_all = register_rest(rec, scene)
t_rest = time.time() - t0
print(f"skeleton: {len(rec['cams'])} photos in {t_skeleton:.0f} s; "
      f"the rest: {len(rec_all['cams']) - len(rec['cams'])} photos in {t_rest:.0f} s")
plot_reconstruction(rec_all, scene, title=f"Trevi: {len(rec_all['cams'])} of {len(members)} photos placed")

### Adding more detail

**Marco:** "I can recognise the fountain, but there are still big gaps."

**Elena:** "So far we've used selected feature points. With the cameras placed, we can match many more pixels."

`densify` follows pixels by optical flow between selected views (up to 64 photos), checks them against the camera
geometry, triangulates them and filters out inconsistent and isolated points. Difficult surfaces and missing
viewpoints can still leave gaps.

In [ ]:
dense = densify(rec_all, scene, n_images=64)
plot_reconstruction(rec_all, scene, dense=dense, title=f"Trevi fountain, dense: {len(dense['pts']):,} points")

**Think about it:** Why can registering the remaining cameras be faster than building the skeleton? What might stop
a photo from registering even when it belongs to the correct landmark group?

**Further reading:** [Photo Tourism](https://phototour.cs.washington.edu/), the incremental SfM that Rome used;
[COLMAP](https://colmap.github.io/) and its paper
[Structure-from-Motion Revisited](https://openaccess.thecvf.com/content_cvpr_2016/papers/Schonberger_Structure-From-Motion_Revisited_CVPR_2016_paper.pdf)
(Schonberger and Frahm 2016), with the homography test for the first pair;
[Hartley and Zisserman](https://www.robots.ox.ac.uk/~vgg/hzbook/), *Multiple View Geometry*, chapter 4 on the DLT
and normalisation; [Szeliski's book](https://szeliski.org/Book/) on SfM and bundle adjustment.

## 5. Putting two maps in the same coordinates

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_st_peters.jpg" alt="St Peter&#x27;s Square and Basilica (Piranesi, 1772)" title="St Peter&#x27;s Square and Basilica (Piranesi, 1772)" width="300" align="right" style="margin: 4px 0 10px 18px">

Marco overlays the Trevi reconstruction on the reference camera positions supplied with the dataset. The two are
shifted, rotated and different in size.

**Marco:** "Have we built it wrong?"

**Elena:** "We need to align the coordinate systems first. Nolli had a compass for north and measured lines for
length. Photos give us no origin, no north and no measured length."

Moving, rotating or scaling the whole scene and its cameras together leaves every image projection unchanged, so a
calibrated reconstruction is fixed only up to a **similarity transform**: seven degrees of freedom in 3D (one scale,
three rotation, three translation), which preserve shape.

### Task 9: `umeyama`

<p align="center" style="text-align: center"><img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/similarity_alignment.png" alt="TAV. VIII: two partial maps of St Peter&#x27;s snapped together by a similarity" title="TAV. VIII: two partial maps of St Peter&#x27;s snapped together by a similarity" width="660" style="display: block; margin: 0 auto"><sub><i>TAV. VIII: two partial maps of St Peter&#x27;s snapped together by a similarity</i></sub></p>

**Marco:** "What can we use to line them up?"

**Elena:** "Points that are the same thing in both maps: camera centres of the same photos. It's like checking a new
survey against an old plan: lay one sheet on the other, slide it until the centres meet, turn it until the shapes
agree, enlarge it until the sizes agree."

We want $s$, $R$ and $\mathbf{t}$ that minimise

$$\sum_i\left\|\mathbf{Y}_i-(sR\mathbf{X}_i+\mathbf{t})\right\|^2.$$

Centre both sets, and take the SVD $U D V^\top$ of the cross-covariance
$\frac{1}{N}\sum_i \mathbf{y}_i \mathbf{x}_i^\top$ of the centred points. Then $R = U S V^\top$ with
$S = \mathrm{diag}(1, 1, \pm 1)$ chosen so that $\det R = +1$ (a rotation, never a reflection);
$s = \mathrm{tr}(DS) / \sigma_x^2$, with $\sigma_x^2$ the mean squared length of the centred `X`; and
$\mathbf{t} = \bar{\mathbf{Y}} - s R \bar{\mathbf{X}}$.

**Your task:** write `umeyama(X, Y)` for two `(N, 3)` arrays whose rows correspond, and return `s`, `R` and `t`; the
method is above. With points as rows, apply it as `Y_est = s * X @ R.T + t`. Library version:
`skimage.transform.SimilarityTransform(dimensionality=3)`.

In [ ]:
def umeyama(X, Y):
    """The similarity transform (scale s, rotation R, shift t) that moves points X onto points Y as well as possible.
    Re-implements skimage.transform.SimilarityTransform(dimensionality=3).estimate(X, Y) (Umeyama 1991).

    Input:  X, Y (N,3) (any array-like), row i of X belongs to row i of Y, N >= 3. Points all on one line are
            fine: the turn about that line is then free, and any R that fits is right.
    Output: (s float, R (3,3) with det R = +1, t (3,)) minimising sum_i |Y_i - (s R X_i + t)|^2.
    """
    # YOUR CODE HERE
    raise NotImplementedError("umeyama")

In [ ]:
test_umeyama()

### Comparing Trevi with the reference

Your transform carries the reconstructed camera centres onto the reference ones. The median error, relative to the
scene size, measures the difference once the arbitrary frame is removed. The reference is another reconstruction
supplied with the data, not a perfect physical survey.

In [ ]:
C_est, C_gt = camera_centres_vs_gt(rec_all, scene)
s, R, t = umeyama(C_est, C_gt)
err = np.linalg.norm(C_gt - (s * C_est @ R.T + t), axis=1)
size = np.median(np.linalg.norm(C_gt - np.median(C_gt, axis=0), axis=1))
print(f"scale {s:.3f}; median camera error {np.median(err) / size:.1%} of the scene size")
plot_alignment(rec_all, scene, s, R, t)

### Joining St Peter's Square and Basilica

**Marco:** "Could we use the same function to join two separate maps?"

**Elena:** "Yes, if they share anchors. Let's try the Square and the Basilica."

The dataset supplies two reference maps of St Peter's, each in its own frame (we do not reconstruct them here).
Photos placed in both are shared anchors, and `umeyama` moves the Basilica map into the Square's frame.

In [ ]:
maps = st_peters_maps(rome)
XA, XB = maps["links_square"], maps["links_basilica"]  # the same anchors, in the two frames
s2, R2, t2 = umeyama(XB, XA)
gap = np.linalg.norm(XA - (s2 * XB @ R2.T + t2), axis=1)
size2 = np.median(np.linalg.norm(XA - np.median(XA, axis=0), axis=1))
print(f"{len(XA)} shared anchors; after the merge they are {np.median(gap) / size2:.1%} of the Square's size apart "
      f"(median)")
plot_merge(maps, s2, R2, t2)

`densify_survey` adds detail to each map from its supplied cameras, as `densify` did for Trevi; the same
`s2, R2, t2` then carry the Basilica's dense cloud into the Square's frame. Check where they overlap: a poor
alignment shows doubled surfaces, or a dome floating off its facade.

In [ ]:
dense_square = densify_survey(rome, "st_peters_square")
dense_basilica = densify_survey(rome, "st_peters_basilica")
plot_dense_merge(dense_square, dense_basilica, s2, R2, t2, title="St Peter's, dense: Square and Basilica as one map")

**Think about it:** `umeyama` fits all supplied anchors. How could a wrongly paired anchor affect the result? How
could RANSAC help before estimating the final transform?

**Further reading:** Umeyama's [paper](https://doi.org/10.1109/34.88573) (1991), with the proof that the sign fix
gives the best rotation; the [pantograph](https://en.wikipedia.org/wiki/Pantograph), the copyist's machine for
enlarging a drawing.

## 6. Ready for the morning demo

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_colosseum.jpg" alt="The Colosseum from outside (Piranesi, 1760)" title="The Colosseum from outside (Piranesi, 1760)" width="300" align="right" style="margin: 4px 0 10px 18px">

Marco opens the component galleries, then the Trevi model and the joined St Peter's clouds.

**Marco:** "I'll start with the original folder. The lighthouse can stay in the first slide."

**Elena:** "Good. Then show how we found the groups, built Trevi and aligned the two St Peter's maps. Show the gaps
as well."

The atlas shows one gallery per major component, with what it shows and how many photos it holds. Other components
are ready for further reconstruction.

In [ ]:
show_atlas(rome, comp, n_components=9)

The steps fit together: visual words make a fast search possible, geometric verification checks the shortlists,
query expansion finds more overlap, and connected components organise the photos. Your matcher builds the
correspondences, the tree selects a skeleton, the homography test picks a starting pair with depth, and a similarity
transform joins coordinate systems.

### What changes with a larger collection?

<img src="https://raw.githubusercontent.com/pikabell/es666-rome-in-a-day/toolbox-63efbeb30581/assets/plate_arch_of_constantine.jpg" alt="The Arch of Constantine (Piranesi, 1771)" title="The Arch of Constantine (Piranesi, 1771)" width="300" align="right" style="margin: 4px 0 10px 18px">

**Marco:** "What if we had a million photos?"

**Elena:** "We'd spread the expensive work across machines and be more careful about which pairs we test."

Feature extraction and pair verification are independent jobs that spread easily; search, storage and reconstruction
need more care. Runtime is not fixed by the photo count: overlap, image quality, search settings and hardware all
matter. The strategy stays: a cheap search decides where expensive geometry is worth doing.

### Things to try

- Change `TOP_K` to 5 or 20. Compare accepted pairs, component sizes and registered cameras.
- Run `show_match_filters(rome, i, j, ratio=0.6)` and `ratio=0.95`. How do the number of matches and the inlier
  fraction change?
- Set `ROUNDS = 0`. Do the components change? Does Trevi register fewer cameras? (Query expansion only proposes
  pairs inside an existing component.)
- Replace tf-idf with normalised raw counts. Which queries gain or lose useful neighbours?
- Reconstruct another component, such as the Pantheon. What makes it easier or harder than Trevi?
- Compare a larger initial reconstruction with the skeleton approach, in runtime and result.
- Compare pairs with high and low homography ratios: is the scene mostly flat, or at several depths?
- Try another image description for retrieval (a tiny thumbnail, a pretrained network). How does the shortlist
  change?

**Further reading:** [COLMAP](https://colmap.github.io/) with [GLOMAP](https://lpanaf.github.io/eccv24_glomap/),
global SfM for large components; [MegaScenes](https://megascenes.github.io/) (2024), over 100,000 SfM
reconstructions of places worldwide; [VGGT](https://vgg-t.github.io/) (2025), a network that predicts cameras and
points from photos.

## 7. Your own photos in 3D

**Marco:** "Would this work on photos I take myself?"

**Elena:** "It should. Walk around a building with your phone, and the same steps that found Trevi will place your
camera at every shot."

Your functions can now rebuild any place you photograph. `rome_from_folder.py` (the setup cell put it next to
this notebook) runs the whole pipeline of this notebook on a set of photos: visual words with your `tfidf_matrix` and
`top_k_neighbours`, your `match_descriptors` on each proposed pair, your `query_expansion` and
`connected_components`, the spanning tree, then the SfM (your `homography_ratio`, with your `homography_dlt`, picks
the first pair) and the dense cloud. It
writes a coloured point cloud that opens in MeshLab, CloudCompare or Blender.

### Task 10: a place you photographed

**Your task:** take 50 or more photos of one place, run the pipeline on them with your functions, and show the result
with `show_results`. Run the two cells below first as they are: they rebuild the Taj Mahal from 150 photos from
Wikimedia Commons (a 47 MB zip in the course repository), which checks that everything works before you use your own
photos. The run takes about 2 minutes on a laptop, a few minutes on Colab. Then set `PHOTOS` to your photos and `OUT`
to a new folder name (say `"my_place_cloud"`), and run both cells again.

**Taking the photos.** Take 50 or more of one place (a building, a statue, a courtyard) while walking around it,
each sharing about half of its view with the next; photos from one spot (turning without walking) give no depth.
JPEG, PNG, WebP or TIFF: iPhones save HEIC unless you set Settings > Camera > Formats > Most Compatible.

**Give them to the next cell.** First put the photos in one folder and zip it (Windows: right-click > Send to >
Compressed (zipped) folder; macOS: right-click > Compress). Then, in quotes, as `PHOTOS`:

1. **Upload the zip** (the simplest, for zips up to a few hundred MB): on Colab, open the Files panel (the folder
   icon on the left), click the upload icon, choose the zip and wait until the upload finishes; then write
   `PHOTOS = "my_photos.zip"` (the zip's name). Files uploaded to Colab are gone when the session ends.
2. **Or a link to the zip** (for big zips, or to reuse it later): put it in a public repository of your own on
   GitHub (Releases > Create a new release, attach the zip, Publish, then copy the link of the zip under Assets), or
   on Google Drive (right-click the zip > Share > Share, change "General access" to **"Anyone with the link"**, Copy
   link), and paste the link.
3. **On your own computer:** put the folder of photos next to this notebook and write `PHOTOS = "my_photos"`.

Give each try its own `OUT` to keep its results.

In [ ]:
# Your functions above must have run. To try the run with the library versions instead, add --reference at the end.
# First as it is (the Taj Mahal), then with your photos and a new OUT.
PHOTOS = "https://github.com/pikabell/es666-rome-in-a-day/releases/download/data-1/taj_mahal.zip"
OUT = "taj_mahal_cloud"  # the folder for the results
%run -i rome_from_folder.py "{PHOTOS}" --out "{OUT}"

The 3D view above is interactive: drag to turn it, scroll to zoom. The next cell shows what the run wrote: the
summary, some of the photos it placed, and the dense cloud from the front and from above. To keep the results,
download `dense.ply` and `view.html` from the `OUT` folder (on Colab: Files panel, right-click > Download; the
session's files go when it ends).

If a run stops or looks wrong, change one setting at a time and run again (add it after `--out "{OUT}"`):

| It says, or you see | What to try |
|---|---|
| "only N of M photos overlap" | `--top-k 20`, and more photos in between the ones you have |
| few photos placed, or "no start pair" | more photos in between, then `--top-k 20` |
| a messy model (wrong pairs) | `--min-inliers 30` (repeated windows and symmetric buildings) |
| the model is bent | `--focal 0.8` for phone photos without EXIF (sent over WhatsApp, say), with `--ignore-exif` if `cameras.csv` says "exif" |
| the cloud is thin | `--dense-images 120`, `--long-side 960` (slower) |
| it is too slow | `--long-side 480 --top-k 5 --rounds 1` |
| the cloud is upside down | `--flip` |

*Example photos: Wikimedia Commons (CC BY, CC BY-SA, CC0 or public domain); the authors are listed in
`attribution.csv` inside the zip.*

In [ ]:
show_results(OUT)

## Three questions before we finish

**Elena:** "We should be ready to explain where this works and where it struggles. Let's check three cases."

Short answers are enough.

### 1. The same place, different lighting

Two photos of the Trevi Fountain, one taken at noon and one at night, may share few visual words.

Why does this happen? Which changes is SIFT designed to handle, and which remain difficult? How might learned local
features or global image descriptors help, and what limitations would remain?

**Your answer:** *(double click to write)*

### 2. Turning without moving

The high-ratio pair A can have many geometric inliers and still be a poor starting pair. Two photos taken from the
same camera centre, with only the orientation changing, cannot provide depth by themselves.

Why can RANSAC find a fundamental matrix with many inliers in this case? Why does that not make its estimated pose
useful for triangulation? Would taking more photos from the same centre solve the problem?

A flat wall photographed from two different positions can also give a ratio near 1. How does this differ from pure
rotation? Is depth fundamentally unavailable, or are the planar correspondences insufficient to determine a unique
`F`?

**Your answer:** *(double click to write)*

### 3. How large is the fountain?

Our reconstruction needs a scale, rotation and translation to align with the reference.

Why are these seven degrees of freedom undetermined by the image projections alone? What extra measurements could
establish metres, orientation and location? If a network predicts 3D points and cameras directly from images, what
would distinguish a plausible size estimate from a measured one?

**Your answer:** *(double click to write)*

## Reading and credits

For more detail: [Building Rome in a Day](https://grail.cs.washington.edu/rome/) for the overall pipeline,
[Photo Tourism](https://phototour.cs.washington.edu/) for SfM from photo collections,
[Lowe's SIFT paper](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf) for local features,
[Video Google](https://www.robots.ox.ac.uk/~vgg/research/vgoogle/) for visual word retrieval, and
[Szeliski's book](https://szeliski.org/Book/) for the geometry and reconstruction methods.

*Plates: G. B. Piranesi, Vedute di Roma (1748 to 1773), and G. B. Nolli, plan of Rome (1748), public domain via
Wikimedia Commons. Diagrams drawn for this notebook; image credits are in
[`CREDITS.md`](https://github.com/pikabell/es666-rome-in-a-day/blob/main/CREDITS.md) in the course repository.
Individual photo attribution is in the dataset's `meta/attribution.csv`.*